In [75]:
import pandas as pd

In [76]:
df = pd.read_csv('user_reviews.csv')

In [77]:
df.head()

,Unnamed: 0,user_id,user_url,reviews
0,0,76561197970982479,http://steamcommunity.com/profiles/76561197970...,"[{'funny': '', 'posted': 'Posted November 5, 2..."
1,1,js41637,http://steamcommunity.com/id/js41637,"[{'funny': '', 'posted': 'Posted June 24, 2014..."
2,2,evcentric,http://steamcommunity.com/id/evcentric,"[{'funny': '', 'posted': 'Posted February 3.',..."
3,3,doctr,http://steamcommunity.com/id/doctr,"[{'funny': '', 'posted': 'Posted October 14, 2..."
4,4,maplemage,http://steamcommunity.com/id/maplemage,"[{'funny': '3 people found this review funny',..."


In [4]:
df = df.drop(columns="Unnamed: 0")

In [10]:
df.shape

(25799, 4)

In [79]:
import ast
def parse_review(reviews):
        return ast.literal_eval(reviews)
    
df['reviews'] = df['reviews'].apply(parse_review)

In [80]:
df['reviews'][1]

[{'funny': '',
  'posted': 'Posted June 24, 2014.',
  'last_edited': '',
  'item_id': '251610',
  'helpful': '15 of 20 people (75%) found this review helpful',
  'recommend': True,
  'review': 'I know what you think when you see this title "Barbie Dreamhouse Party" but do not be intimidated by it\'s title, this is easily one of my GOTYs. You don\'t get any of that cliche game mechanics that all the latest games have, this is simply good core gameplay. Yes, you can\'t 360 noscope your friends, but what you can do is show them up with your bad ♥♥♥ dance moves and put them to shame as you show them what true fashion and color combinations are.I know this game says for kids but, this is easily for any age range and any age will have a blast playing this.8/8'},
 {'funny': '',
  'posted': 'Posted September 8, 2013.',
  'last_edited': '',
  'item_id': '227300',
  'helpful': '0 of 1 people (0%) found this review helpful',
  'recommend': True,
  'review': "For a simple (it's actually not all th

# Endpoints

* Endpoint 1: def userdata( User_id : str ): Debe devolver cantidad de dinero gastado por el usuario, el porcentaje de recomendación en base a reviews.recommend y cantidad de items.

#### Calculo el porcentaje de recomendación en base a reviews.recommend

In [9]:
import re
def calculate_recommendation_percentage(review_list):
    total_helpful = 0
    total_count = 0

    for review_dict in review_list:
        helpful = review_dict.get('helpful', '')
        match = re.search(r'(\d+) of (\d+) people', helpful)
        if match:
            helpful_count = int(match.group(1))
            total_count += int(match.group(2))
            total_helpful += helpful_count
            
    if total_count > 0:
        return (total_helpful / total_count) * 100
    else:
        return 0

df['porcentaje_recomendacion'] = df['reviews'].apply(calculate_recommendation_percentage)


       porcentaje_recomendacion
0                      0.000000
1                     64.000000
2                      0.000000
3                     83.333333
4                     81.250000
...                         ...
25794                  0.000000
25795                  0.000000
25796                100.000000
25797                  0.000000
25798                 50.000000

[25799 rows x 1 columns]


In [11]:
df.head()

,user_id,user_url,reviews,porcentaje_recomendacion
0,76561197970982479,http://steamcommunity.com/profiles/76561197970...,"[{'funny': '', 'posted': 'Posted November 5, 2...",0.000000
1,js41637,http://steamcommunity.com/id/js41637,"[{'funny': '', 'posted': 'Posted June 24, 2014...",64.000000
2,evcentric,http://steamcommunity.com/id/evcentric,"[{'funny': '', 'posted': 'Posted February 3.',...",0.000000
3,doctr,http://steamcommunity.com/id/doctr,"[{'funny': '', 'posted': 'Posted October 14, 2...",83.333333
4,maplemage,http://steamcommunity.com/id/maplemage,"[{'funny': '3 people found this review funny',...",81.250000


In [13]:
df['porcentaje_recomendacion'] = df['porcentaje_recomendacion'].apply(lambda x: str(round(x)) + '%')

In [12]:
df_endpoint1 = pd.read_csv("df_endpoint1.csv")

In [15]:
df_endpoint1['porcentaje_recomendacion'] = df['porcentaje_recomendacion']

In [16]:
df_endpoint1.head()

,user_id,items_count,gasto_total,porcentaje_recomendacion
0,76561197970982479,277,3419.32,0%
1,js41637,888,8394.37,64%
2,evcentric,137,1584.90,0%
3,Riot-Punch,328,3284.47,83%
4,doctr,541,6660.58,81%


In [17]:
nuevo_orden = ['user_id', 'gasto_total', 'porcentaje_recomendacion', 'items_count']
df_endpoint1 = df_endpoint1[nuevo_orden]

In [18]:
df_endpoint1.head()

,user_id,gasto_total,porcentaje_recomendacion,items_count
0,76561197970982479,3419.32,0%,277
1,js41637,8394.37,64%,888
2,evcentric,1584.90,0%,137
3,Riot-Punch,3284.47,83%,328
4,doctr,6660.58,81%,541


In [19]:
df_endpoint1.to_csv("df_endpoint1.csv", index=False)

#### * Endpoint2: def countreviews( YYYY-MM-DD y YYYY-MM-DD : str ): Cantidad de usuarios que realizaron reviews entre las fechas dadas y, el porcentaje de recomendación de los mismos en base a reviews.recommend.

In [32]:
import calendar
def mes_a_numero(nombre_mes):
    mes_numero = list(calendar.month_name).index(nombre_mes.capitalize())
    return mes_numero

In [130]:
import re

def fecha_reviews(reviews_list):
    fecha_review = []
    for review_dict in reviews_list:
        recommend = review_dict.get('recommend')
        if recommend is True:
            recommend_add = 1
        else:
            recommend_add = 0
            
        posted = review_dict.get('posted', '')
        match_date = re.search(r'Posted (\b\w+\b) (\d{1,2}), (\d{4}).', posted)
        if match_date:
            month = match_date.group(1)
            day = match_date.group(2)
            year = match_date.group(3)
            fecha = f"{year}-{month}-{day}"
            fecha_review.append([recommend_add, fecha])
            
    return fecha_review


In [128]:
df["date_recommend"] = df['reviews'].apply(fecha_reviews)

In [131]:
df_endpoint2 = df[["user_id","date_recommend"]]
df_endpoint2 = df_endpoint2[df_endpoint2['date_recommend'].apply(lambda x: bool(x))]

In [144]:
df_endpoint2.to_csv("df_endpoint2.csv", index=False)

In [19]:
df = pd.read_csv("df_endpoint2.csv")

In [51]:
df_endpoint2["date_recommend"][4253]

"[[1, '2015-July-4'], [0, '2015-May-8'], [1, '2015-February-7'], [1, '2015-September-16'], [0, '2015-May-8']]"

In [54]:
from datetime import datetime
 
def count_reviews(start_date: str, end_date: str):
    # Convert start_date and end_date to datetime objects
    start_date = datetime.strptime(start_date, '%Y-%m-%d')
    end_date = datetime.strptime(end_date, '%Y-%m-%d')

    # Filter the DataFrame based on the date range
    filtered_df = df[df['date_recommend'].apply(lambda x: any(start_date <= datetime.strptime(date_str, '%Y-%B-%d') <= end_date and is_positive for is_positive, date_str in x))]

    # Calculate the count of users and the percentage of recommendations
    user_count = len(filtered_df)
    total_recommendations = sum(recommend for sublist in filtered_df['date_recommend'] for _, date_str in sublist if start_date <= datetime.strptime(date_str, '%Y-%B-%d') <= end_date)
    total_reviews = sum(1 for sublist in filtered_df['date_recommend'] for _, date_str in sublist if start_date <= datetime.strptime(date_str, '%Y-%B-%d') <= end_date)
    
    if total_reviews == 0:
        recommendation_percentage = 0.0
    else:
        recommendation_percentage = (total_recommendations / total_reviews) * 100

    return {
        "user_count": user_count,
        "recommendation_percentage": recommendation_percentage
    }

In [55]:
count_reviews("2005-12-12", "2008-12-12")

ValueError: not enough values to unpack (expected 2, got 1)

In [65]:
from datetime import datetime

def count_reviews(start_date: str, end_date: str, df):
    # Convert start_date and end_date to datetime objects
    start_date = datetime.strptime(start_date, '%Y-%m-%d')
    end_date = datetime.strptime(end_date, '%Y-%m-%d')

    # Filter the DataFrame based on the date range
    def filter_date_range(date_list):
        for is_positive, date_str in date_list:
            date = datetime.strptime(date_str, '%Y-%B-%d')
            if start_date <= date <= end_date and is_positive:
                return True
        return False

    filtered_df = df[df['date_recommend'].apply(filter_date_range)]

    # Calculate the count of users and the percentage of recommendations
    user_count = len(filtered_df)
    
    total_recommendations = sum(sum(is_positive for is_positive, _ in sublist) for sublist in filtered_df['date_recommend'])
    
    total_reviews = sum(1 for sublist in filtered_df['date_recommend'] if filter_date_range(sublist))

    if total_reviews == 0:
        recommendation_percentage = 0.0
    else:
        recommendation_percentage = (total_recommendations / total_reviews) * 100

    return {
        "user_count": user_count,
        "recommendation_percentage": recommendation_percentage
    }


In [72]:
count_reviews("2012-05-05", "2020-08-08",df)

{'user_count': 21826, 'recommendation_percentage': 201.37908915971775}

#### Endpoint sentiment analysis

* def sentiment_analysis( año : int ): Según el año de lanzamiento, se devuelve una lista con la cantidad de registros de reseñas de usuarios que se encuentren categorizados con un análisis de sentimiento.

Feature Engineering: En el dataset user_reviews se incluyen reseñas de juegos hechos por distintos usuarios. Debes crear la columna 'sentiment_analysis' aplicando análisis de sentimiento con NLP con la siguiente escala: debe tomar el valor '0' si es malo, '1' si es neutral y '2' si es positivo. Esta nueva columna debe reemplazar la de user_reviews.review para facilitar el trabajo de los modelos de machine learning y el análisis de datos. De no ser posible este análisis por estar ausente la reseña escrita, debe tomar el valor de 1.

In [91]:
df["reviews"][4]

[{'funny': '3 people found this review funny',
  'posted': 'Posted April 15, 2014.',
  'last_edited': '',
  'item_id': '211420',
  'helpful': '35 of 43 people (81%) found this review helpful',
  'recommend': True,
  'review': 'Git gud'},
 {'funny': '1 person found this review funny',
  'posted': 'Posted December 23, 2013.',
  'last_edited': '',
  'item_id': '211820',
  'helpful': '12 of 16 people (75%) found this review helpful',
  'recommend': True,
  'review': "It's like Terraria, you play for 9 hours straight, get endgame armour then stop playing until the next update."},
 {'funny': '2 people found this review funny',
  'posted': 'Posted March 14, 2014.',
  'last_edited': '',
  'item_id': '730',
  'helpful': '5 of 5 people (100%) found this review helpful',
  'recommend': True,
  'review': 'Hold shift to win, Hold CTRL to lose.'},
 {'funny': '',
  'posted': 'Posted July 11, 2013.',
  'last_edited': '',
  'item_id': '204300',
  'helpful': 'No ratings yet',
  'recommend': True,
  'rev

In [81]:
df_reviews = pd.DataFrame(df["reviews"])

In [82]:
from textblob import TextBlob
def analyze_sentiment(reviews_list):
    sentiments = []
    for review_dict in reviews_list:
        analysis = TextBlob(review_dict['review'])
        polarity = analysis.sentiment.polarity
        if polarity < 0:
            sentiments.append(0)  # Negative
        elif polarity == 0:
            sentiments.append(1)  # Neutral
        else:
            sentiments.append(2)  # Positive
    return sentiments

In [83]:
df_reviews['sentiment_analysis'] = df_reviews['reviews'].apply(analyze_sentiment)

In [87]:
df_reviews.iloc[1]

reviews               [{'funny': '', 'posted': 'Posted June 24, 2014...
sentiment_analysis                                            [2, 0, 0]
Name: 1, dtype: object

In [201]:
df_reviews["reviews"][37]

[{'funny': '',
  'posted': 'Posted January 6, 2015.',
  'last_edited': '',
  'item_id': '213670',
  'helpful': 'No ratings yet',
  'recommend': True,
  'review': 'My character is a Jew named ♥♥♥♥♥♥♥♥♥ that is wearing a KKK costume.10/10, IGN'}]

In [103]:
import re
def sentiment_analysis(year):
    # Filter out rows with empty lists in 'reviews' column
    df_filtered = df_reviews[df_reviews['reviews'].apply(lambda x: isinstance(x, list) and len(x) > 0)]

    # Extract the year using regex for non-empty reviews
    pattern = r'Posted (\w+ \d{1,2}, \d{4}).'
    def extract_year(review):
        match = re.search(pattern, review[0]['posted'])
        if match:
            return int(match.group(1).split()[-1])
        else:
            return None

    df_filtered['posted_year'] = df_filtered['reviews'].apply(extract_year)

    # Filter reviews for the specified year
    filtered_reviews = df_filtered[df_filtered['posted_year'] == year]

    # Flatten the 'sentiment_analysis' lists and count sentiment categories
    sentiment_counts = filtered_reviews['sentiment_analysis'].explode().value_counts().to_dict()

    # Map sentiment values to their labels
    sentiment_labels = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
    sentiment_counts = {sentiment_labels[key]: value for key, value in sentiment_counts.items()}

    return sentiment_counts

In [110]:
years_of_interest = list(range(2009, 2016))
# Create a dictionary to store results
results_dict = {'Year': [], 'Sentiment Analysis': []}

# Loop through the years and collect results
for year in years_of_interest:
    sentiment_result = sentiment_analysis(year)
    results_dict['Year'].append(year)
    results_dict['Sentiment Analysis'].append(sentiment_result)

# Convert the results to a DataFrame
results_df = pd.DataFrame(results_dict)

C:\Users\jazmin\AppData\Local\Temp\ipykernel_14092\779080910.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered['posted_year'] = df_filtered['reviews'].apply(extract_year)
C:\Users\jazmin\AppData\Local\Temp\ipykernel_14092\779080910.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered['posted_year'] = df_filtered['reviews'].apply(extract_year)
C:\Users\jazmin\AppData\Local\Temp\ipykernel_14092\779080910.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slic

In [112]:
results_df.to_csv("df_endpoint6.csv",index=False)

In [113]:
results_df

,Year,Sentiment Analysis
0,2009,{}
1,2010,"{'Positive': 15, 'Negative': 5, 'Neutral': 1}"
2,2011,"{'Positive': 164, 'Negative': 50, 'Neutral': 35}"
3,2012,"{'Positive': 423, 'Negative': 119, 'Neutral': 80}"
4,2013,"{'Positive': 3198, 'Neutral': 1149, 'Negative'..."
5,2014,"{'Positive': 12493, 'Neutral': 4701, 'Negative..."
6,2015,"{'Positive': 10757, 'Negative': 4426, 'Neutral..."
